## Kinesis Write from Databricks
simulate streaming from a static DataFrame, sending one row every 10 seconds to Kinesis. This is common for testing.

Spark Structured Streaming doesn’t natively support “one row every N seconds” from a static DataFrame, so we have to manually stream it using a loop. But it needs to be done row by row using .toPandas() or .collect(), because df is static.

In [0]:
import boto3
import json
import time

# Load AWS credentials from Databricks secrets
aws_access_key = dbutils.secrets.get("my-aws-secrets", "aws-access-key").strip()
aws_secret_key = dbutils.secrets.get("my-aws-secrets", "aws-secret-key").strip()
aws_region = "us-west-2"

# Create Kinesis client
kinesis_client = boto3.client(
    "kinesis",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key
)

# Read your parquet file into Spark DataFrame
df = spark.read.parquet("/Volumes/smart_claims_dev/00_lading/raw_telematics")

# Collect rows to driver (small dataset only!)
rows = df.collect()[::-1]  # reverse order if needed

# Send each row to Kinesis, one at a time
for row in rows:
    # Convert Row to dictionary
    row_dict = row.asDict()
    
    # Send to Kinesis
    response = kinesis_client.put_record(
        StreamName="telematics-stream-mgm",
        Data=json.dumps(row_dict),  # serialize as JSON
        PartitionKey=row_dict.get("chassis_no", "key1")  # use chassis_no as key
    )
    
    print("Sent row:", row_dict, "| Response:", response)
    time.sleep(10)  # wait 10 seconds before sending next row

In [0]:
len(rows)

## The corretc a way to do it on Databricks is with all dataframe at once

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import MapType, StringType

kinesis_config = {
    "streamName": "telematics-stream-mgm",
    "region": "us-west-2",
    "serviceCredential": "mgm-service-credentials",
    "initialPosition": "earliest",
    "checkpointLocation": "/tmp/checkpoints/telematics"  # required for streaming
}

payload_schema = MapType(StringType(), StringType())

# Read your parquet as static DataFrame
df = spark.read.parquet("/Volumes/smart_claims_dev/00_lading/raw_telematics")

# If you need streaming semantics, convert to streaming DF (optional, depends on use case)
# For static DF -> streaming write works too, just processes all rows as one batch

df.writeStream \
    .format("kinesis") \
    .options(**kinesis_config) \
    .start() \
    .awaitTermination(60)  # stops after 60 seconds

## Step 1 — Make sure you’re using string, not bytes

In [0]:
##If running in Databricks, SparkSession exists already as spark.
## If running locally, uncomment and adjust:
# from pyspark.sql import SparkSession
# spark = Session.builder. \
# appName("Movies101"). V
# getOrCreate()

import boto3
import json

# Load AWS credentials from Databricks secrets
aws_access_key = dbutils.secrets.get("my-aws-secrets", "aws-access-key").strip()
aws_secret_key = dbutils.secrets.get("my-aws-secrets", "aws-secret-key").strip()

# (Optional) define region
aws_region = "us-west-2"

## Step 2 — Use the boto3 client

In [0]:
kinesis_client = boto3.client(
    "kinesis",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key
)

## Step 3 — Test listing streams

In [0]:
streams = kinesis_client.list_streams()
print("Available streams:", streams.get("StreamNames"))

## Step 4 — Sending a test record

In [0]:
import json

response = kinesis_client.put_record(
    StreamName="telematics-stream-mgm",
    Data=json.dumps({"msg": "Hello from Databricks, nice to meet you!"}),
    PartitionKey="key1"
)

print("Sent record:", response)

## Step 5 — Get shards from the stream

In [0]:
stream_name = "telematics-stream-mgm"

stream_desc = kinesis_client.describe_stream(StreamName=stream_name)
shards = stream_desc['StreamDescription']['Shards']

# Print shards
for shard in shards:
    print(f"ShardId: {shard['ShardId']}")

## Step 6 — Get a shard iterator
ShardIteratorType options:
* 'TRIM_HORIZON' → start from oldest record
* 'LATEST' → only new records
* 'AT_SEQUENCE_NUMBER' / 'AFTER_SEQUENCE_NUMBER' → specific sequence

In [0]:
shard_id = shards[3]['ShardId']  # just take the first shard for testing
display(shard_id)
shard_iterator_response = kinesis_client.get_shard_iterator(
    StreamName=stream_name,
    ShardId=shard_id,
    ShardIteratorType='LATEST'  # reads from the start of the stream
)
display(shard_iterator_response['ShardIterator'])
shard_iterator = shard_iterator_response['ShardIterator']

## Step 7 — Read records
Each record in Records has:
* Data → the message you sent (JSON string in our case)
* PartitionKey → what you set (key1)
* SequenceNumber → Kinesis internal sequence number

In [0]:
response = kinesis_client.get_records(
    ShardIterator=shard_iterator,
    Limit=10
)

records = response['Records']

# Loop over records safely
for r in records:
    try:
        # Decode bytes to string
        data_str = r['Data'].decode('utf-8')  # usually utf-8
        # Load JSON
        data_json = json.loads(data_str)
        display(data_json)  # use Databricks display
    except (UnicodeDecodeError, json.JSONDecodeError) as e:
        display(f"Skipping record due to decode error: {e}")

## Step 5 — Iterate for new messages
For continuous reads, you can loop:

**Note**: In serverless Databricks, avoid very tight loops — they can spike cluster usage. Poll every 1–2 seconds for testing.

In [0]:
import time

shard_iterator = shard_iterator  # initial iterator

while True:
    response = kinesis_client.get_records(
        ShardIterator=shard_iterator,
        Limit=10
    )
    records = response['Records']
    for r in records:
        print(json.loads(r['Data']))

    shard_iterator = response['NextShardIterator']
    time.sleep(1)  # wait before polling again